In [1]:

#################################################################################################################################################################################################################################################################################
#CONEXION AL ORACLE

import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config
import decouple 
config = decouple.AutoConfig(' ')

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM CMPRS10", con=connection2)

connection2.close()




In [2]:
base2.shape

(139168, 40)

In [3]:
a=base2[base2.duplicated(subset=["tipdocidenpercod","perasisdocidennum"])]
a

,tipdocidenpercod,perasisdocidennum,perasispllaremcod,grupocupcod,perasisprocolcod,perasisproapepat,perasisproapemat,perasisproprinom,perasisprosegnom,condtrabcod,...,perusuinifirdig,peripinifirdig,perusumodsusfirdig,peripmodsusfirdig,perusumodinifirdig,peripmodinifirdig,perlocfecnac,perlocserextflg,perespcod1,perespcod2


In [4]:
base2.columns

Index(['tipdocidenpercod', 'perasisdocidennum', 'perasispllaremcod',
       'grupocupcod', 'perasisprocolcod', 'perasisproapepat',
       'perasisproapemat', 'perasisproprinom', 'perasisprosegnom',
       'condtrabcod', 'perasisvacinifec', 'perasisvacfinfec',
       'perasisestregcod', 'perflgfirdig', 'perfecinifirdig',
       'perfecmodsusfirdig', 'perfecmodinifirdig', 'peremail1', 'peremail2',
       'pertel1', 'pertel2', 'perfecmodctcdat', 'perusumodctcdat',
       'perasisprousucrecod', 'perasisprofeccre', 'perasisproipcre',
       'perasisprousumodcod', 'perasisprofecmod', 'perasisproipmod',
       'peripmodctcdat', 'perusuinifirdig', 'peripinifirdig',
       'perusumodsusfirdig', 'peripmodsusfirdig', 'perusumodinifirdig',
       'peripmodinifirdig', 'perlocfecnac', 'perlocserextflg', 'perespcod1',
       'perespcod2'],
      dtype='object')

In [5]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER="postgres"
DB_PASSWORD="AKindOfMagic"
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST="10.0.1.228"
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

query_count_before = "SELECT COUNT(*) FROM cmprs10"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla cmprs10 antes de la inserción: {cant_antes}")

#connection3.execute('CREATE TEMPORARY TABLE tmp_cmprs10 ()')
base2.to_sql(name='tmp_cmprs10', con=engine3, if_exists='replace', index=False)

Cantidad de filas en la tabla cmprs10 antes de la inserción: 135970


168

In [6]:


#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL cmprs10 FALSO CON LO OBTENIDO DEL ORACLE

query="""
ALTER TABLE tmp_cmprs10 
ALTER COLUMN tipdocidenpercod TYPE CHARACTER (1),
ALTER COLUMN perasisdocidennum TYPE CHARACTER (10),
ALTER COLUMN perasispllaremcod TYPE NUMERIC (10),
ALTER COLUMN grupocupcod TYPE CHARACTER (2),
ALTER COLUMN perasisprocolcod TYPE CHARACTER (10),
ALTER COLUMN perasisproapepat TYPE CHARACTER (30),
ALTER COLUMN perasisproapemat TYPE CHARACTER (30),
ALTER COLUMN perasisproprinom TYPE CHARACTER (20),
ALTER COLUMN perasisprosegnom TYPE CHARACTER (20),
ALTER COLUMN condtrabcod TYPE CHARACTER (1),
ALTER COLUMN perasisvacinifec TYPE DATE,
ALTER COLUMN perasisvacfinfec TYPE DATE,
ALTER COLUMN perasisestregcod TYPE CHARACTER (1),
ALTER COLUMN perflgfirdig TYPE CHARACTER (1),
ALTER COLUMN perfecinifirdig TYPE DATE,
ALTER COLUMN perfecmodsusfirdig TYPE DATE,
ALTER COLUMN perfecmodinifirdig TYPE DATE,
ALTER COLUMN peremail1 TYPE CHARACTER (100),
ALTER COLUMN peremail2 TYPE CHARACTER (100),
ALTER COLUMN pertel1 TYPE CHARACTER (10),
ALTER COLUMN pertel2 TYPE CHARACTER (10),
ALTER COLUMN perfecmodctcdat TYPE DATE,
ALTER COLUMN perusumodctcdat TYPE CHARACTER (10),
ALTER COLUMN perasisprousucrecod TYPE CHARACTER (10),
ALTER COLUMN perasisprofeccre TYPE DATE,
ALTER COLUMN perasisproipcre TYPE CHARACTER (15),
ALTER COLUMN perasisprousumodcod TYPE CHARACTER (10),
ALTER COLUMN perasisprofecmod TYPE DATE,
ALTER COLUMN perasisproipmod TYPE CHARACTER (15),
ALTER COLUMN peripmodctcdat TYPE CHARACTER (15),
ALTER COLUMN perusuinifirdig TYPE CHARACTER (10),
ALTER COLUMN peripinifirdig TYPE CHARACTER (15),
ALTER COLUMN perusumodsusfirdig TYPE CHARACTER (10),
ALTER COLUMN peripmodsusfirdig TYPE CHARACTER (15),
ALTER COLUMN perusumodinifirdig TYPE CHARACTER (10),
ALTER COLUMN peripmodinifirdig TYPE CHARACTER (15);
"""

c1= text(query)
connection3.execute(c1)


In [7]:


#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL cmprs10 FALSO CON LO OBTENIDO DEL ORACLE

query="""
TRUNCATE TABLE cmprs10;

INSERT INTO cmprs10 
(tipdocidenpercod, perasisdocidennum, perasispllaremcod,
grupocupcod, perasisprocolcod, perasisproapepat,
perasisproapemat, perasisproprinom, perasisprosegnom,
condtrabcod, perasisvacinifec, perasisvacfinfec,
perasisestregcod, perflgfirdig, perfecinifirdig,
perfecmodsusfirdig, perfecmodinifirdig, peremail1, peremail2,
pertel1, pertel2, perfecmodctcdat, perusumodctcdat,
perasisprousucrecod, perasisprofeccre, perasisproipcre,
perasisprousumodcod, perasisprofecmod, perasisproipmod,
peripmodctcdat, perusuinifirdig, peripinifirdig,
perusumodsusfirdig, peripmodsusfirdig, perusumodinifirdig,
peripmodinifirdig) 
SELECT 
tipdocidenpercod, perasisdocidennum, perasispllaremcod,
grupocupcod, perasisprocolcod, perasisproapepat,
perasisproapemat, perasisproprinom, perasisprosegnom,
condtrabcod, perasisvacinifec, perasisvacfinfec,
perasisestregcod, perflgfirdig, perfecinifirdig,
perfecmodsusfirdig, perfecmodinifirdig, peremail1, peremail2,
pertel1, pertel2, perfecmodctcdat, perusumodctcdat,
perasisprousucrecod, perasisprofeccre, perasisproipcre,
perasisprousumodcod, perasisprofecmod, perasisproipmod,
peripmodctcdat, perusuinifirdig, peripinifirdig,
perusumodsusfirdig, peripmodsusfirdig, perusumodinifirdig,
peripmodinifirdig

FROM tmp_cmprs10  t
WHERE NOT EXISTS (
    SELECT 1 
    FROM cmprs10
    WHERE cmprs10.tipdocidenpercod = t.tipdocidenpercod and cmprs10.tipdocidenpercod IS NOT NULL AND
    cmprs10.perasisdocidennum = t.perasisdocidennum and cmprs10.perasisdocidennum IS NOT NULL 
) ;
"""

c1= text(query)
connection3.execute(c1)


In [8]:

# #BORRAMOS LAS TABLAS
# query2="""
# DROP TABLE tmp_cmprs10;
# DELETE FROM cmprs10 WHERE tipdocidenpercod ='';
# """
# c2= text(query2)
# cursor=connection3.execute(c2)
# cant_despues = cursor.fetchone()[0]
# print(f"Cantidad de filas en la tabla cmprs10 después de la inserción: {cant_despues}")
# print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")
base2 = pd.read_sql_query(f"SELECT * FROM cmprs10", con=connection3)
# connection3.execute(c2)

In [9]:

base2 = pd.read_sql_query(f"SELECT * FROM cmprs10", con=connection3)

connection3.close()


In [10]:
base2.columns

Index(['tipdocidenpercod', 'perasisdocidennum', 'perasispllaremcod',
       'grupocupcod', 'perasisprocolcod', 'perasisproapepat',
       'perasisproapemat', 'perasisproprinom', 'perasisprosegnom',
       'condtrabcod', 'perasisvacinifec', 'perasisvacfinfec',
       'perasisestregcod', 'perflgfirdig', 'perfecinifirdig',
       'perfecmodsusfirdig', 'perfecmodinifirdig', 'peremail1', 'peremail2',
       'pertel1', 'pertel2', 'perfecmodctcdat', 'perusumodctcdat',
       'perasisprousucrecod', 'perasisprofeccre', 'perasisproipcre',
       'perasisprousumodcod', 'perasisprofecmod', 'perasisproipmod',
       'peripmodctcdat', 'perusuinifirdig', 'peripinifirdig',
       'perusumodsusfirdig', 'peripmodsusfirdig', 'perusumodinifirdig',
       'peripmodinifirdig'],
      dtype='object')

In [11]:
#################################################################################################################################################################################################################################################################################


#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine4 = create_engine(cadena4)
connection4 = engine4.connect()


base2.to_sql(name='tmp_cmprs10', con=engine4, if_exists='replace', index=False)

168

In [12]:

#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS

query="""

ALTER TABLE tmp_cmprs10 
ALTER COLUMN tipdocidenpercod TYPE CHARACTER (1),
ALTER COLUMN perasisdocidennum TYPE CHARACTER (10),
ALTER COLUMN perasispllaremcod TYPE NUMERIC (10),
ALTER COLUMN grupocupcod TYPE CHARACTER (2),
ALTER COLUMN perasisprocolcod TYPE CHARACTER (10),
ALTER COLUMN perasisproapepat TYPE CHARACTER (30),
ALTER COLUMN perasisproapemat TYPE CHARACTER (30),
ALTER COLUMN perasisproprinom TYPE CHARACTER (20),
ALTER COLUMN perasisprosegnom TYPE CHARACTER (20),
ALTER COLUMN condtrabcod TYPE CHARACTER (1),
ALTER COLUMN perasisvacinifec TYPE DATE,
ALTER COLUMN perasisvacfinfec TYPE DATE,
ALTER COLUMN perasisestregcod TYPE CHARACTER (1),
ALTER COLUMN perflgfirdig TYPE CHARACTER (1),
ALTER COLUMN perfecinifirdig TYPE DATE,
ALTER COLUMN perfecmodsusfirdig TYPE DATE,
ALTER COLUMN perfecmodinifirdig TYPE DATE,
ALTER COLUMN peremail1 TYPE CHARACTER (100),
ALTER COLUMN peremail2 TYPE CHARACTER (100),
ALTER COLUMN pertel1 TYPE CHARACTER (10),
ALTER COLUMN pertel2 TYPE CHARACTER (10),
ALTER COLUMN perfecmodctcdat TYPE DATE,
ALTER COLUMN perusumodctcdat TYPE CHARACTER (10),
ALTER COLUMN perasisprousucrecod TYPE CHARACTER (10),
ALTER COLUMN perasisprofeccre TYPE DATE,
ALTER COLUMN perasisproipcre TYPE CHARACTER (15),
ALTER COLUMN perasisprousumodcod TYPE CHARACTER (10),
ALTER COLUMN perasisprofecmod TYPE DATE,
ALTER COLUMN perasisproipmod TYPE CHARACTER (15),
ALTER COLUMN peripmodctcdat TYPE CHARACTER (15),
ALTER COLUMN perusuinifirdig TYPE CHARACTER (10),
ALTER COLUMN peripinifirdig TYPE CHARACTER (15),
ALTER COLUMN perusumodsusfirdig TYPE CHARACTER (10),
ALTER COLUMN peripmodsusfirdig TYPE CHARACTER (15),
ALTER COLUMN perusumodinifirdig TYPE CHARACTER (10),
ALTER COLUMN peripmodinifirdig TYPE CHARACTER (15);
"""



c1= text(query)
connection4.execute(c1)


In [13]:
query="""
UPDATE dim_personal 
SET 
nombre = CONCAT(TRIM(t.perasisproapepat), ' ', TRIM(t.perasisproapemat), ' ', TRIM(t.perasisproprinom), ' ', TRIM(t.perasisprosegnom))


FROM tmp_cmprs10 t 
WHERE dim_personal.tip_doc = t.tipdocidenpercod AND dim_personal.num_doc=t.perasisdocidennum AND dim_personal.gru_ocu=t.grupocupcod AND dim_personal.tip_doc  IS NOT NULL;


INSERT INTO dim_personal (tip_doc, num_doc, nombre, gru_ocu,cod_pla,colegio,condicion) 
SELECT tipdocidenpercod, perasisdocidennum, CONCAT(TRIM(t.perasisproapepat), ' ', TRIM(t.perasisproapemat), ' ', TRIM(t.perasisproprinom), ' ', TRIM(t.perasisprosegnom)), grupocupcod,perasispllaremcod,perasisprocolcod,condtrabcod    

FROM tmp_cmprs10 t
WHERE NOT EXISTS (
    SELECT 1 
    FROM dim_personal 
    WHERE dim_personal.tip_doc = t.tipdocidenpercod AND dim_personal.num_doc = t.perasisdocidennum AND dim_personal.gru_ocu=t.grupocupcod
);

DROP TABLE tmp_cmprs10;
"""


c1= text(query)
connection4.execute(c1)

#BORRAMOS LAS TABLAS
query2=f"""
SELECT COUNT(*) FROM dim_personal;
"""

c2= text(query2)
cursor=connection4.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla dim_personal después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

Cantidad de filas en la tabla dim_personal después de la inserción: 140847
La cantidad de filas insertadas fue de: 4877
